<a href="https://colab.research.google.com/github/niofrequency/google-collab_image/blob/main/UPSCALE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Upscale & detail (images and videos)

Makes pictures and videos bigger and sharper, for every tab of the website: Image Generator, Face Swap, Head Swap and
Image to Video results (press **Send to Upscale** in their full-size viewers), or any picture / video you upload.

- **Pictures**: 2×, 3× or 4× with **4x-UltraSharp** (crisp, good for generated images) or **Real-ESRGAN x4plus** (photos),
  plus optional **face restore** (GFPGAN v1.4) with a strength slider.
- **Videos**: every frame to 720p / 1080p / 1440p with the same upscalers, the **sound kept as it was**, and optional
  **smoother motion** (2× frame rate, ffmpeg motion interpolation — e.g. 16 fps → 32 fps).

Runs on any GPU (T4 is fine; L4 / A100 are faster for videos). Models: ~1 GB, downloaded in cell 1.

1. **Runtime → Change runtime type → T4 GPU** (or L4 / A100).
2. Run **cell 1** (install + models), **cell 2** (saves the server program) and **cell 3** (prints the `READY:` URL).
3. Paste the URL into the website's **Upscale** tab.

**Cell 4** shows the server's log. Upscaling works on the pixels only — it doesn't redraw content; for a "creative"
upscale that adds new detail, use the Image Generator's **Refiner** (BigLust) instead.

In [ ]:
#@title 1. Install + download the models (~1 GB)
!pip install -q spandrel facexlib fastapi uvicorn python-multipart
!which ffmpeg >/dev/null || apt-get install -qqy ffmpeg

import os, sys, urllib.request
MODEL_DIR = "/content/upscale_models"
os.makedirs(MODEL_DIR, exist_ok=True)
FILES = {
    "4x-UltraSharp.pth": "https://huggingface.co/lokCX/4x-Ultrasharp/resolve/main/4x-UltraSharp.pth",
    "RealESRGAN_x4plus.pth": "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth",
    "RealESRGAN_x2plus.pth": "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth",
    "GFPGANv1.4.pth": "https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.4.pth",
    "facexlib/detection_Resnet50_Final.pth": "https://github.com/xinntao/facexlib/releases/download/v0.1.0/detection_Resnet50_Final.pth",
    "facexlib/parsing_parsenet.pth": "https://github.com/xinntao/facexlib/releases/download/v0.2.2/parsing_parsenet.pth",
}
for name, url in FILES.items():
    path = os.path.join(MODEL_DIR, name)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    if not os.path.exists(path):
        print(f"Downloading {name}...")
        urllib.request.urlretrieve(url, path + ".part")
        os.replace(path + ".part", path)
print("✅ Models ready:", ", ".join(FILES))
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "No GPU: Runtime -> Change runtime type -> T4 GPU (it works on the CPU too, but slowly)"

In [ ]:
%%writefile /content/upscale_server.py
# Cell 2: the upscale server, saved as its own program (cell 3 runs it and restarts it if it crashes).
import base64
import gc
import io
import os
import re
import tempfile
import threading
import time
import traceback
import uuid
from typing import Optional

os.environ.setdefault("UPSCALE_MODEL_DIR", "/content/upscale_models")
# ---------------------------------------------------------------- upscalers, face restore, video
# Upscalers (spandrel loads them, like ComfyUI does). The 4x models are also used for 2x: upscale 4x, then shrink
# to 2x with Lanczos, which is sharper than a 2x model.
import os
import shutil
import subprocess
import tempfile
import urllib.request

import numpy as np
import torch
from PIL import Image

MODEL_DIR = os.environ.get("UPSCALE_MODEL_DIR", "/content/upscale_models")
UPSCALERS = {
    "ultrasharp": ("4x-UltraSharp.pth", "https://huggingface.co/lokCX/4x-Ultrasharp/resolve/main/4x-UltraSharp.pth"),
    "realesrgan": ("RealESRGAN_x4plus.pth", "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth"),
    "realesrgan_x2": ("RealESRGAN_x2plus.pth", "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth"),
}
GFPGAN = ("GFPGANv1.4.pth", "https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.4.pth")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
_models = {}


def model_file(name, url):
    os.makedirs(MODEL_DIR, exist_ok=True)
    path = os.path.join(MODEL_DIR, name)
    if not os.path.exists(path):
        print(f"Downloading {name}...", flush=True)
        urllib.request.urlretrieve(url, path + ".part")
        os.replace(path + ".part", path)
    return path


def load_model(key):
    """Loads an upscaler (or GFPGAN) once; fp16 on the GPU when the model supports it."""
    if key not in _models:
        from spandrel import ModelLoader
        name, url = GFPGAN if key == "gfpgan" else UPSCALERS[key]
        m = ModelLoader().load_from_file(model_file(name, url)).eval().to(DEVICE)
        if DEVICE == "cuda" and m.supports_half:
            m = m.half()
        _models[key] = m
    return _models[key]


def to_tensor(pil, model):
    t = torch.from_numpy(np.array(pil.convert("RGB"))).permute(2, 0, 1).float().div(255)[None]
    return t.to(DEVICE, dtype=next(model.model.parameters()).dtype)


def to_pil(t):
    return Image.fromarray(t[0].float().clamp(0, 1).permute(1, 2, 0).mul(255).round().byte().cpu().numpy())


@torch.inference_mode()
def run_tiled(model, img, tile=512, pad=24):
    """Upscales a [1,3,H,W] tensor in overlapping tiles, so big pictures fit in GPU memory."""
    _, _, H, W = img.shape
    s = model.scale
    out = torch.zeros((1, 3, H * s, W * s), dtype=img.dtype, device=img.device)
    for y in range(0, H, tile):
        for x in range(0, W, tile):
            th, tw = min(tile, H - y), min(tile, W - x)
            y0, x0 = max(0, y - pad), max(0, x - pad)
            y1, x1 = min(H, y + th + pad), min(W, x + tw + pad)
            o = model(img[:, :, y0:y1, x0:x1])
            out[:, :, y * s:(y + th) * s, x * s:(x + tw) * s] = \
                o[:, :, (y - y0) * s:(y - y0 + th) * s, (x - x0) * s:(x - x0 + tw) * s]
    return out


def upscale(pil, model_key="ultrasharp", scale=2.0, tile=512):
    """Upscales a picture by `scale` (1-4). Returns a PIL image."""
    model = load_model(model_key)
    out = to_pil(run_tiled(model, to_tensor(pil, model), tile=tile))
    w, h = round(pil.width * scale), round(pil.height * scale)
    return out if out.size == (w, h) else out.resize((w, h), Image.Resampling.LANCZOS)


_face_helper = {}


@torch.inference_mode()
def restore_faces(pil, strength=0.7):
    """GFPGAN v1.4 on every face (facexlib finds and aligns them, then pastes them back with a soft mask).
    strength 0 = original face, 1 = fully restored."""
    import cv2
    from facexlib.utils.face_restoration_helper import FaceRestoreHelper
    if "h" not in _face_helper:
        _face_helper["h"] = FaceRestoreHelper(1, face_size=512, crop_ratio=(1, 1), det_model="retinaface_resnet50",
                                              save_ext="png", use_parse=True, device=DEVICE,
                                              model_rootpath=os.path.join(MODEL_DIR, "facexlib"))
    helper, gfpgan = _face_helper["h"], load_model("gfpgan")
    helper.clean_all()
    bgr = cv2.cvtColor(np.array(pil.convert("RGB")), cv2.COLOR_RGB2BGR)
    helper.read_image(bgr)
    helper.get_face_landmarks_5(only_center_face=False, resize=640, eye_dist_threshold=5)
    helper.align_warp_face()
    if not helper.cropped_faces:
        return pil, 0
    for face in helper.cropped_faces:
        t = torch.from_numpy(cv2.cvtColor(face, cv2.COLOR_BGR2RGB)).permute(2, 0, 1).float().div(255)[None]
        t = t.to(DEVICE, dtype=next(gfpgan.model.parameters()).dtype)
        out_t = gfpgan.model(t * 2 - 1)[0]   # GFPGAN works in -1..1, like the original code
        restored = cv2.cvtColor(np.array(to_pil((out_t.float() + 1) / 2)), cv2.COLOR_RGB2BGR)
        if strength < 1:
            restored = cv2.addWeighted(restored, strength, face, 1 - strength, 0)
        helper.add_restored_face(restored)
    helper.get_inverse_affine(None)
    out = helper.paste_faces_to_input_image(upsample_img=bgr)
    return Image.fromarray(cv2.cvtColor(out.astype(np.uint8), cv2.COLOR_BGR2RGB)), len(helper.cropped_faces)


def probe(path):
    """fps, width, height, has_audio of a video (ffprobe)."""
    import json
    info = json.loads(subprocess.run(["ffprobe", "-v", "error", "-show_streams", "-of", "json", path],
                                     capture_output=True, text=True, check=True).stdout)
    v = next(s for s in info["streams"] if s["codec_type"] == "video")
    num, den = (v.get("r_frame_rate") or "16/1").split("/")
    return float(num) / float(den or 1), int(v["width"]), int(v["height"]), any(s["codec_type"] == "audio" for s in info["streams"])


def upscale_video(src_path, dst_path, model_key="realesrgan", target_height=720, interpolate=False,
                  faces=False, face_strength=0.7, on_progress=None):
    """Upscales every frame to `target_height` (even width), keeps the sound, optionally doubles the frame rate
    (ffmpeg motion interpolation). Returns info."""
    fps, w, h, has_audio = probe(src_path)
    scale = target_height / h
    out_w, out_h = int(round(w * scale / 2)) * 2, int(round(target_height / 2)) * 2
    work = tempfile.mkdtemp(prefix="upscale-")
    try:
        subprocess.run(["ffmpeg", "-v", "error", "-i", src_path, os.path.join(work, "f%06d.png")], check=True)
        frames = sorted(f for f in os.listdir(work) if f.endswith(".png"))
        enc = subprocess.Popen(["ffmpeg", "-v", "error", "-y", "-f", "rawvideo", "-pix_fmt", "rgb24", "-s", f"{out_w}x{out_h}",
                                "-r", f"{fps}", "-i", "-", "-c:v", "libx264", "-preset", "medium", "-crf", "17",
                                "-pix_fmt", "yuv420p", os.path.join(work, "video.mp4")], stdin=subprocess.PIPE)
        for i, name in enumerate(frames):
            frame = Image.open(os.path.join(work, name)).convert("RGB")
            big = upscale(frame, model_key, scale) if scale > 1 else frame
            if faces:
                big, _ = restore_faces(big, face_strength)
            if big.size != (out_w, out_h):
                big = big.resize((out_w, out_h), Image.Resampling.LANCZOS)
            enc.stdin.write(np.array(big).tobytes())
            if on_progress:
                on_progress(i + 1, len(frames))
        enc.stdin.close()
        if enc.wait() != 0:
            raise RuntimeError("ffmpeg could not write the video")
        video = os.path.join(work, "video.mp4")
        if interpolate:   # 2x frames: motion-compensated interpolation (smoother motion)
            subprocess.run(["ffmpeg", "-v", "error", "-y", "-i", video, "-vf",
                            f"minterpolate=fps={fps * 2:g}:mi_mode=mci:mc_mode=aobmc:vsbmc=1",
                            "-c:v", "libx264", "-preset", "medium", "-crf", "17", "-pix_fmt", "yuv420p",
                            os.path.join(work, "smooth.mp4")], check=True)
            video = os.path.join(work, "smooth.mp4")
        if has_audio:   # the original sound track, unchanged
            subprocess.run(["ffmpeg", "-v", "error", "-y", "-i", video, "-i", src_path, "-map", "0:v:0", "-map", "1:a:0",
                            "-c:v", "copy", "-c:a", "copy", "-shortest", dst_path], check=True)
        else:
            shutil.copyfile(video, dst_path)
        return {"frames": len(frames), "fps": fps * (2 if interpolate else 1), "width": out_w, "height": out_h, "audio": has_audio}
    finally:
        shutil.rmtree(work, ignore_errors=True)


# ---------------------------------------------------------------- API for the website
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])

JOBS = {}
GPU_LOCK = threading.Lock()
MAX_OUTPUT_SIDE = 8192          # pictures: longest side of the result
VIDEO_HEIGHTS = (480, 720, 1080, 1440)
MODEL_NAMES = {"ultrasharp": "4x-UltraSharp", "realesrgan": "Real-ESRGAN x4plus"}


class UpscaleRequest(BaseModel):
    image: Optional[str] = None          # data URL of a picture ...
    video: Optional[str] = None          # ... or of a video (mp4 / webm)
    model: str = "ultrasharp"            # "ultrasharp" or "realesrgan"
    scale: float = 2.0                   # pictures: 1-4
    target_height: int = 1080            # videos: 480 / 720 / 1080 / 1440
    face_restore: bool = False
    face_strength: float = 0.7           # 0 = original face ... 1 = fully restored
    interpolate: bool = False            # videos: 2x frame rate (smoother motion)


def decode_data_url(value):
    m = re.match(r"^data:([^;,]*)(;base64)?,", value or "")
    return base64.b64decode(value[m.end():] if m else value), (m.group(1) if m else "")


def new_job(**extra):
    now = time.time()
    for jid in [j for j, v in JOBS.items() if v["status"] in ("done", "error") and now - v["created"] > 1800]:
        JOBS.pop(jid, None)
    job_id = uuid.uuid4().hex[:12]
    JOBS[job_id] = {"status": "queued", "created": now, "error": None, "progress": 0, "message": "Queued", **extra}
    return job_id


# ---------------- Memory guard ----------------
# Colab kills the server when the runtime's RAM (12.7 GB on a T4) runs out. When free RAM gets low, this frees the
# caches this server keeps for speed (they're rebuilt when needed) and hands freed memory back to the system.
RAM_LOW_GB = 3.0        # before a job
RAM_CRITICAL_GB = 2.0   # after a job


def ram_available_gb():
    try:
        with open("/proc/meminfo") as fh:
            for line in fh:
                if line.startswith("MemAvailable:"):
                    return int(line.split()[1]) / 1024 / 1024
    except OSError:
        pass
    return 99.0


def memory_guard(after=False):
    limit = RAM_CRITICAL_GB if after else RAM_LOW_GB
    free = ram_available_gb()
    if free >= limit:
        return
    import ctypes
    import gc as _gc
    freed = []
    try:
        freed = free_caches() or []
    except Exception as e:
        print(f"Memory guard: could not free caches ({e})", flush=True)
    try:
        import comfy.model_management as _mm
        for fn in ("cleanup_models_gc", "cleanup_models"):
            try:
                getattr(_mm, fn)()
            except Exception:
                pass
    except Exception:
        pass
    _gc.collect()
    try:
        import torch as _torch
        _torch.cuda.empty_cache()
    except Exception:
        pass
    try:
        ctypes.CDLL("libc.so.6").malloc_trim(0)   # give freed memory back to the system
    except Exception:
        pass
    print(f"Memory guard ({free:.1f} GB RAM free {'after' if after else 'before'} a job): freed "
          f"{', '.join(freed) or 'unused memory'} — {ram_available_gb():.1f} GB free now", flush=True)


def free_caches():
    freed = []
    if _models:
        _models.clear(); freed.append("upscaler models")
    if _face_helper:
        _face_helper.clear(); freed.append("face finder")
    return freed


def run_job(job_id, req):
    job = JOBS[job_id]
    with GPU_LOCK:
        memory_guard()
        job.update(status="running", message="Starting")
        t0 = time.time()
        try:
            key = req.model if req.model in MODEL_NAMES else "ultrasharp"
            if req.video:
                raw, mime = decode_data_url(req.video)
                height = min(VIDEO_HEIGHTS, key=lambda h: abs(h - int(req.target_height or 1080)))
                with tempfile.TemporaryDirectory() as tmp:
                    src = os.path.join(tmp, "in" + (".webm" if "webm" in mime else ".mp4"))
                    dst = os.path.join(tmp, "out.mp4")
                    open(src, "wb").write(raw)

                    def progress(i, n):
                        job.update(progress=round(3 + 90 * i / n), message=f"Upscaling frame {i}/{n}")
                    info = upscale_video(src, dst, key, height, bool(req.interpolate), bool(req.face_restore),
                                         float(req.face_strength), on_progress=progress)
                    if req.interpolate:
                        job["message"] = "Smoothing the motion"
                    job.update(video="data:video/mp4;base64," + base64.b64encode(open(dst, "rb").read()).decode(), **info)
                print(f"[{job_id}] video {info} with {MODEL_NAMES[key]} in {time.time() - t0:.0f}s", flush=True)
            elif req.image:
                raw, _ = decode_data_url(req.image)
                from PIL import ImageOps
                pil = ImageOps.exif_transpose(Image.open(io.BytesIO(raw))).convert("RGB")
                scale = max(1.0, min(float(req.scale or 2), 4.0, MAX_OUTPUT_SIDE / max(pil.size)))
                job["message"] = f"Upscaling {pil.width}x{pil.height} x{scale:g}"
                out = upscale(pil, key, scale)
                faces = 0
                if req.face_restore:
                    job["message"] = "Restoring faces"
                    out, faces = restore_faces(out, max(0.0, min(float(req.face_strength), 1.0)))
                buf = io.BytesIO()
                big = out.width * out.height > 4_500_000          # big results as high-quality JPEG (PNG would be 20+ MB)
                out.save(buf, "JPEG" if big else "PNG", **({"quality": 95} if big else {}))
                job.update(image=f"data:image/{'jpeg' if big else 'png'};base64," + base64.b64encode(buf.getvalue()).decode(),
                           width=out.width, height=out.height, faces=faces)
                print(f"[{job_id}] picture {pil.size} -> {out.size} with {MODEL_NAMES[key]}, {faces} face(s) restored, "
                      f"{time.time() - t0:.1f}s", flush=True)
            else:
                raise ValueError("Send a picture or a video")
            job.update(status="done", progress=100, message="Done", seconds=round(time.time() - t0, 1))
        except Exception as e:
            traceback.print_exc()
            job.update(status="error", error=str(e) or type(e).__name__)
        finally:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            memory_guard(after=True)


@app.get("/")
def root():
    return {"status": "ok", "mode": "upscale"}


@app.get("/health")
def health():
    return {"status": "ok", "loaded": True, "device": DEVICE}


@app.get("/user_models")
def user_models():
    gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    return {"model_type": "upscale", "models": MODEL_NAMES, "video_heights": list(VIDEO_HEIGHTS),
            "supports_video": True, "supports_faces": True, "gpu": gpu, "max_output_side": MAX_OUTPUT_SIDE}


@app.post("/upscale")
def upscale_endpoint(req: UpscaleRequest):
    job_id = new_job(kind="video" if req.video else "image")
    threading.Thread(target=run_job, args=(job_id, req), daemon=True).start()
    return {"job_id": job_id, "status": "queued"}


@app.get("/job/{job_id}")
def get_job(job_id: str):
    job = JOBS.get(job_id)
    if not job:
        return {"status": "error", "error": "Unknown job (the server restarted). Try again."}
    return {**{k: v for k, v in job.items() if k != "created"}, "job_id": job_id}


if __name__ == "__main__":
    import uvicorn
    print(f"Upscale server starting on {DEVICE}...", flush=True)
    for k in ("ultrasharp", "realesrgan", "gfpgan"):   # load the models now so the first job is quick
        try:
            load_model(k)
        except Exception as e:
            print(f"Could not load {k}: {e}", flush=True)
    uvicorn.run(app, host="127.0.0.1", port=8000)


In [ ]:
#@title 3. Start the upscale server + Cloudflare tunnel
import os
import re
import subprocess
import sys
import time
import urllib.request

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "fastapi", "uvicorn", "python-multipart"],
    check=True,
)

SERVER_SCRIPT = "/content/upscale_server.py"
SERVER_LOG = "/content/upscale_server.log"
TUNNEL_LOG = "/content/cloudflared.log"
RUNNER = "/content/run_upscale_server.sh"
PORT = 8000
CLOUDFLARED = "/usr/local/bin/cloudflared"

subprocess.run(["pkill", "-f", "run_upscale_server.sh"], capture_output=True)
subprocess.run(["pkill", "-f", "upscale_server.py"], capture_output=True)
subprocess.run(["pkill", "-f", "cloudflared"], capture_output=True)
time.sleep(1)

if not os.path.exists(CLOUDFLARED):
    print("Downloading cloudflared...")
    subprocess.run(
        [
            "wget",
            "-q",
            "-O",
            CLOUDFLARED,
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        ],
        check=True,
    )
    os.chmod(CLOUDFLARED, 0o755)

# The server runs in a loop: if it ever crashes, it starts again on the same port, so the
# website's Backend URL keeps working (after the model has loaded again, a few minutes).
with open(RUNNER, "w") as f:
    f.write(
        "#!/bin/bash\n"
        "while true; do\n"
        f"  {sys.executable} {SERVER_SCRIPT}\n"
        "  echo \"[runner] server stopped (exit code $?) - restarting in 5 s\"\n"
        "  sleep 5\n"
        "done\n"
    )
os.chmod(RUNNER, 0o755)

print("Starting upscale server...")
server_env = {**os.environ, "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"}
with open(SERVER_LOG, "w") as log:
    server_proc = subprocess.Popen(
        ["bash", RUNNER],
        env=server_env,
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

print("Waiting for model to load and Uvicorn to bind port 8000...")
model_ready = False
for _ in range(450):  # loading the ~32 GB model can take several minutes
    time.sleep(2)
    if server_proc.poll() is not None:
        print("❌ Server process exited early. Check log below:")
        break
    if os.path.exists(SERVER_LOG):
        with open(SERVER_LOG) as f:
            content = f.read()
        if "Application startup complete" in content:
            try:
                with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=3) as resp:
                    if resp.status == 200 and b'"loaded":true' in resp.read().replace(b" ", b""):
                        print("✅ Model loaded and /health is ready")
                        model_ready = True
                        break
            except Exception:
                pass

if not model_ready:
    print("⚠️ Model did not finish loading in time. Last 30 log lines:")
    with open(SERVER_LOG) as f:
        print("".join(f.readlines()[-30:]))
    raise RuntimeError("Upscale server failed to become ready.")

print("Starting Cloudflare tunnel...")
# Its output goes to a file. (Reading it through a pipe that nobody empties later makes cloudflared
# freeze once the pipe is full, and then the website gets "Failed to fetch" / "Load failed".)
tunnel_log = open(TUNNEL_LOG, "w")
tunnel_proc = subprocess.Popen(
    [CLOUDFLARED, "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{PORT}"],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

TUNNEL_URL = None
t0 = time.time()
while time.time() - t0 < 90 and TUNNEL_URL is None:
    time.sleep(1)
    with open(TUNNEL_LOG) as f:
        m = re.search(r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com", f.read())
    if m:
        TUNNEL_URL = m.group(0)

if TUNNEL_URL:
    print("\n" + "=" * 60)
    print(f"READY: {TUNNEL_URL}")
    print("=" * 60)
    print("\n→ Copy the URL above and paste it into the Upscale tab on your website.")
else:
    print("❌ Could not get Cloudflare URL. Try re-running this cell.")

In [ ]:
#@title 4. Show the latest server and tunnel logs (for troubleshooting)
!tail -40 /content/upscale_server.log
!echo; echo '--- tunnel ---'; tail -10 /content/cloudflared.log